In [10]:
# Install required packages
!pip install langgraph langchain-google-genai langchain-core

In [11]:
import os
from typing import TypedDict
from langgraph.graph import StateGraph, END
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import HumanMessage

# Set your Gemini API key
os.environ["GOOGLE_API_KEY"] = ""

# Initialize Gemini 2.5 Flash
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash")

In [12]:
# USE CASE 1: Telecom SIM Activation & Fraud Verification

In [13]:
# Define the state that flows through the telecom workflow
class TelecomState(TypedDict):
    customer_name: str
    aadhaar_number: str
    pan_number: str
    location: str
    previous_sim_requests: int
    kyc_status: str
    fraud_risk: str
    final_decision: str

In [14]:
# Step 1 - KYC Verification Tool
# Validates Aadhaar and PAN documents and returns verification status
def kyc_verification_tool(state: TelecomState) -> TelecomState:
    aadhaar = state["aadhaar_number"]
    pan = state["pan_number"]

    # Basic validation: Aadhaar should be 12 digits, PAN should be 10 chars
    aadhaar_valid = len(aadhaar) == 12 and aadhaar.isdigit()
    pan_valid = len(pan) == 10 and pan[:5].isalpha() and pan[5:9].isdigit() and pan[9].isalpha()

    if aadhaar_valid and pan_valid:
        kyc_status = "VERIFIED"
    elif not aadhaar_valid and not pan_valid:
        kyc_status = "INVALID_DOCUMENT"
    else:
        kyc_status = "PENDING_VERIFICATION"

    print(f"[KYC] Status: {kyc_status}")
    return {**state, "kyc_status": kyc_status}

In [15]:
# Step 2 - Fraud Detection using Gemini
# Gemini analyzes location, sim request history, and KYC status to classify risk
def fraud_detection_node(state: TelecomState) -> TelecomState:
    prompt = f"""
    You are a telecom fraud detection system. Analyze the following customer data and classify the fraud risk.

    Customer Location: {state['location']}
    Number of Previous SIM Requests: {state['previous_sim_requests']}
    KYC Status: {state['kyc_status']}

    Based on this information, classify the fraud risk as one of:
    - LOW_RISK
    - MEDIUM_RISK
    - HIGH_RISK

    Respond with ONLY the classification label, nothing else.
    """

    response = llm.invoke([HumanMessage(content=prompt)])
    fraud_risk = response.content.strip()

    print(f"[Fraud Detection] Risk: {fraud_risk}")
    return {**state, "fraud_risk": fraud_risk}

In [16]:
# Step 3 - Final Activation Decision
# Based on fraud risk classification, decide the SIM activation outcome
def activation_decision_node(state: TelecomState) -> TelecomState:
    risk = state["fraud_risk"]

    if "LOW_RISK" in risk:
        decision = "SIM Activated"
    elif "MEDIUM_RISK" in risk:
        decision = "Manual Review Required"
    else:
        decision = "Application Rejected"

    print(f"[Activation] Decision: {decision}")
    return {**state, "final_decision": decision}

In [17]:
# Build the telecom sequential workflow graph
telecom_graph = StateGraph(TelecomState)

# Add nodes
telecom_graph.add_node("kyc_verification", kyc_verification_tool)
telecom_graph.add_node("fraud_detection", fraud_detection_node)
telecom_graph.add_node("activation_decision", activation_decision_node)

# Define sequential edges
telecom_graph.set_entry_point("kyc_verification")
telecom_graph.add_edge("kyc_verification", "fraud_detection")
telecom_graph.add_edge("fraud_detection", "activation_decision")
telecom_graph.add_edge("activation_decision", END)

# Compile the graph
telecom_app = telecom_graph.compile()

In [18]:
# Run the telecom workflow with a sample customer
telecom_input = {
    "customer_name": "Raj Kumar",
    "aadhaar_number": "123456789012",
    "pan_number": "ABCDE1234F",
    "location": "Mumbai",
    "previous_sim_requests": 2,
    "kyc_status": "",
    "fraud_risk": "",
    "final_decision": ""
}

print("=== Telecom SIM Activation Workflow ===")
result = telecom_app.invoke(telecom_input)
print("\n--- Final Result ---")
print(f"Customer   : {result['customer_name']}")
print(f"KYC Status : {result['kyc_status']}")
print(f"Fraud Risk : {result['fraud_risk']}")
print(f"Decision   : {result['final_decision']}")

=== Telecom SIM Activation Workflow ===
[KYC] Status: VERIFIED
[Fraud Detection] Risk: LOW_RISK
[Activation] Decision: SIM Activated

--- Final Result ---
Customer   : Raj Kumar
KYC Status : VERIFIED
Fraud Risk : LOW_RISK
Decision   : SIM Activated


In [19]:
# USE CASE 2: Healthcare Appointment Prioritization Workflow

In [20]:
# Define the state that flows through the healthcare workflow
class HealthcareState(TypedDict):
    patient_name: str
    age: int
    fever: float          # temperature in Celsius
    oxygen_level: int     # SpO2 percentage
    heart_rate: int       # BPM
    symptom_duration: int # days
    existing_conditions: str
    severity: str
    priority: str
    consultation_type: str

In [21]:
# Step 1 - Symptom Severity Tool
# Analyzes vitals and returns a severity classification
def symptom_severity_tool(state: HealthcareState) -> HealthcareState:
    fever = state["fever"]
    oxygen = state["oxygen_level"]
    heart_rate = state["heart_rate"]
    duration = state["symptom_duration"]

    # Critical if oxygen is very low or fever is very high or heart rate is extreme
    if oxygen < 90 or fever > 104 or heart_rate > 130 or heart_rate < 40:
        severity = "CRITICAL"
    # Moderate if values are concerning but not dangerous
    elif oxygen < 95 or fever > 101 or heart_rate > 100 or duration > 5:
        severity = "MODERATE"
    else:
        severity = "STABLE"

    print(f"[Symptom Check] Severity: {severity}")
    return {**state, "severity": severity}

In [22]:
# Step 2 - Gemini Medical Prioritization
# Gemini evaluates severity, age, and existing conditions to assign priority
def medical_prioritization_node(state: HealthcareState) -> HealthcareState:
    prompt = f"""
    You are a hospital triage system. Evaluate the following patient information and assign a consultation priority.

    Symptom Severity: {state['severity']}
    Patient Age: {state['age']}
    Existing Conditions: {state['existing_conditions']}

    Based on this information, classify the priority as one of:
    - EMERGENCY
    - PRIORITY_CONSULTATION
    - REGULAR_CONSULTATION

    Respond with ONLY the classification label, nothing else.
    """

    response = llm.invoke([HumanMessage(content=prompt)])
    priority = response.content.strip()

    print(f"[Medical Priority] Priority: {priority}")
    return {**state, "priority": priority}

In [23]:
# Step 3 - Final Consultation Assignment
# Assigns the patient to the correct consultation type based on priority
def consultation_assignment_node(state: HealthcareState) -> HealthcareState:
    priority = state["priority"]

    if "EMERGENCY" in priority:
        consultation = "ICU / Emergency Room"
    elif "PRIORITY_CONSULTATION" in priority:
        consultation = "Specialist Doctor"
    else:
        consultation = "General Physician"

    print(f"[Consultation] Assigned to: {consultation}")
    return {**state, "consultation_type": consultation}

In [24]:
# Build the healthcare sequential workflow graph
healthcare_graph = StateGraph(HealthcareState)

# Add nodes
healthcare_graph.add_node("symptom_check", symptom_severity_tool)
healthcare_graph.add_node("medical_priority", medical_prioritization_node)
healthcare_graph.add_node("consultation_assignment", consultation_assignment_node)

# Define sequential edges
healthcare_graph.set_entry_point("symptom_check")
healthcare_graph.add_edge("symptom_check", "medical_priority")
healthcare_graph.add_edge("medical_priority", "consultation_assignment")
healthcare_graph.add_edge("consultation_assignment", END)

# Compile the graph
healthcare_app = healthcare_graph.compile()

In [25]:
# Run the healthcare workflow with a sample patient
healthcare_input = {
    "patient_name": "Meena Iyer",
    "age": 68,
    "fever": 103.5,
    "oxygen_level": 91,
    "heart_rate": 115,
    "symptom_duration": 3,
    "existing_conditions": "Diabetes, Hypertension",
    "severity": "",
    "priority": "",
    "consultation_type": ""
}

print("=== Healthcare Appointment Prioritization Workflow ===")
result = healthcare_app.invoke(healthcare_input)
print("\n--- Final Result ---")
print(f"Patient          : {result['patient_name']}")
print(f"Severity         : {result['severity']}")
print(f"Priority         : {result['priority']}")
print(f"Consultation     : {result['consultation_type']}")

=== Healthcare Appointment Prioritization Workflow ===
[Symptom Check] Severity: MODERATE
[Medical Priority] Priority: PRIORITY_CONSULTATION
[Consultation] Assigned to: Specialist Doctor

--- Final Result ---
Patient          : Meena Iyer
Severity         : MODERATE
Priority         : PRIORITY_CONSULTATION
Consultation     : Specialist Doctor
